# 09 — Author SLM-SQL + CSC baseline

Canonical workflow: **09 → 10 → 11**. Do not run Qwen SFT notebooks 02–08 for this experiment.
Use the author's SFT + GRPO generator and SFT + GRPO CSC merger. Prompts reconstruct Appendix E semantically; this is not a claim of reproducing the paper's exact score.

Before running: attach BIRD **dev**. Attach BIRD **train** as well to build LoRA data; set `BUILD_CSC_TRAIN_CACHE=0` for a dev-only author baseline. Set `CSC_TRAIN_DIR` to the folder with `train.json` and `train_databases/`; set `CSC_EVAL_DIR` to the folder with `dev.json` and `dev_databases/`. Defaults are a **pilot**, not full evaluation. Set `CSC_PILOT=0` **before first config creation** for the full protocol. Use a new `CSC_CONFIG` and `CSC_EXPERIMENT_DIR` when changing protocol.

GPU stages are controlled by `RUN_CSC_GPU=1`; completed matching stages reuse artifacts without loading a model. `BUILD_CSC_TRAIN_CACHE=0` runs the author evaluation baseline only. Preserve the entire experiment folder and JSON config between sessions. See `docs/CSC_LORA.md`.


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

configured = os.environ.get('SLM_PRJ_PATH')
candidates = ([Path(configured)] if configured else []) + [Path.cwd(), *Path.cwd().parents, Path('/kaggle/working/SLM-PRJ')]
REPO_DIR = next((p.resolve() for p in candidates if (p / 'research/slm_sql').is_dir()), None)
if REPO_DIR is None:
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
    subprocess.run(['git', 'clone', os.environ.get('SLM_PRJ_REPO_URL', 'https://github.com/minhadcaz/SLMSQL_project.git'), str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))


In [ ]:
# Run once in an inference session. Enable only if dependencies are not installed.
INSTALL_DEPENDENCIES = os.environ.get('INSTALL_CSC_DEPS', '0') == '1'
if INSTALL_DEPENDENCIES:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '-r', str(REPO_DIR / 'requirements-csc-inference.txt')], check=True)


In [ ]:
from scripts.csc_notebook import configure, run
CONFIG, cfg = configure(REPO_DIR)
SEED = int(os.environ.get('CSC_SEED', '42'))
RUN_GPU = os.environ.get('RUN_CSC_GPU', '0') == '1'
BUILD_TRAIN_CACHE = os.environ.get('BUILD_CSC_TRAIN_CACHE', '1') == '1'
print({'config': str(CONFIG), 'experiment': cfg['experiment_dir'], 'seed': SEED, 'run_gpu': RUN_GPU})
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


## Freeze splits and inspect work before using GPU

CPU only: checks database separation, SQLite files and gold SQL compilation, hashes input data and stores schemas. A dev-only preparation can be extended with train/validation later while preserving evaluation candidates. Validation databases are held out from adapter training; the author's upstream model may already have seen BIRD train.


In [ ]:
prepare_flags = [] if BUILD_TRAIN_CACHE else ['--evaluation-only']
run(REPO_DIR, CONFIG, 'prepare', *prepare_flags)
run(REPO_DIR, CONFIG, 'plan', '--seed', str(SEED))


## Evaluate the original generator + original CSC

Generation, execution and merge run in separate subprocesses so generator and merger do not remain loaded together. Default full protocol uses 64 generator samples, CSC only when there are at least two executable result groups, then 8 merge samples. Pilot uses 4/2.


In [ ]:
if RUN_GPU:
    run(REPO_DIR, CONFIG, 'baseline', '--seed', str(SEED))
else:
    print('GPU stages not started. Set RUN_CSC_GPU=1 after inspecting plan.')


## Cache training/validation candidates once

These candidates come from the **training benchmark** partitions, never final dev. No merger call is needed to build LoRA examples: top-2 candidates are checked using train gold execution. Disable this section when only evaluating the author baseline.


In [ ]:
if RUN_GPU and BUILD_TRAIN_CACHE:
    for split in ('train', 'validation'):
        run(REPO_DIR, CONFIG, 'generate', '--split', split, '--seed', str(SEED))
        run(REPO_DIR, CONFIG, 'execute', '--split', split, '--seed', str(SEED))
else:
    print('Training candidate stages not started.')


In [ ]:
k = cfg['generation']['evaluation_samples']
report_path = Path(cfg['experiment_dir']) / 'reports' / f'evaluation-seed{SEED}-k{k}.baseline.json'
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(json.dumps(report['summary'], indent=2))
else:
    print('No baseline score exists yet; no score is fabricated.')
